In [1]:
import torch, os, sys
import numpy as np
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_math_sdp(True)
import inspect
os.environ['CUDA_VISIBLE_DEVICES'] = '3'
import pandas as pd
from fast_transformers.builders import TransformerEncoderBuilder
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from fast_transformers.masking import FullMask, LengthMask

root_dir = os.path.dirname(os.getcwd())
sys.path.append(root_dir)
import pdb
import torch.nn.functional as F
import random

In [2]:
n_layers = 2
num_heads = 1
embed_dim = 32
n_hid = 128
dropout = 0.1

In [3]:
torch.manual_seed(0)
# Create the builder for our transformers
builder = TransformerEncoderBuilder.from_kwargs(
    n_layers=n_layers,
    n_heads=num_heads,
    query_dimensions=embed_dim // num_heads,
    value_dimensions=embed_dim // num_heads,
    feed_forward_dimensions=n_hid,
    dropout=dropout
)

# Build a transformer with softmax attention
builder.attention_type = "full"
softmax_model = builder.get().to('cuda')

# Build a transformer with linear attention
builder.attention_type = "linear"
linear_model = builder.get().to('cuda')


In [4]:
def generate_associative_recall_batch(seq_len=10, batch_size=32, vocab_size=10, device='cuda'):
    """Generate a batch of associative recall training data with discrete tokens"""
    # Generate random key-value pairs from a limited vocabulary
    # Generate random token IDs
    keys = torch.randint(0, vocab_size // 2, (batch_size, vocab_size), device=device)
    values = torch.randint(vocab_size // 2, vocab_size, (batch_size, vocab_size), device=device)
    
    token_embeddings = torch.randn(vocab_size, embed_dim, device=device)  # Random embedding matrix

    kv_indices = torch.randint(0, len(keys), (batch_size, seq_len), device=device)
    keys_embedded = token_embeddings[keys[kv_indices]]
    values_embedded = token_embeddings[values[kv_indices]]
    
    # Randomly select query indices
    query_indices = torch.randint(0, seq_len, (batch_size,), device=device)
    queries = keys_embedded[torch.arange(batch_size), query_indices]  # [batch, embed_dim]
    
    # True targets are the values corresponding to the queries
    targets = values[torch.arange(batch_size), query_indices]  # [batch]
    
    # Prepare input sequence: concatenate keys and values, then append query
    input_seq = torch.stack([keys_embedded, values_embedded], dim=2)  # [batch, seq_len, 2, embed_dim]
    input_seq = input_seq.view(batch_size, seq_len*2, embed_dim)  # [batch, seq_len*2, embed_dim]
    input_seq = torch.cat([input_seq, queries.unsqueeze(1)], dim=1)  # [batch, seq_len*2+1, embed_dim]

    return input_seq, targets


In [5]:

def train_model(model, seq_len, vocab_size=10, n_epochs=1000, batch_size=1024, device='cuda'):
    """Train model on associative recall classification task"""
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=0.1)
    criterion = torch.nn.CrossEntropyLoss()
    losses = []
    
    for epoch in range(n_epochs):
        model.train()
        input_seq, targets = generate_associative_recall_batch(seq_len, batch_size, vocab_size, device)
        
        # Create attention mask
        attention_mask = FullMask(seq_len*2 + 1)
        
        # Forward pass
        optimizer.zero_grad()

        output = model(input_seq, attn_mask=attention_mask)  # [batch, seq_len*2+1, vocab_size]
        predictions = output[:, -1]  # Take last token predictions [batch, vocab_size]
        
        # Compute loss
        loss = criterion(predictions, targets)
        # Compute accuracy
        acc = (predictions.argmax(dim=-1) == targets).float().mean()
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}/{n_epochs}, Loss: {loss.item():.6f}, Acc: {acc.item():.6f}")
    
    return np.mean(losses[-10:])  # Return average of last 10 losses

In [6]:
def evaluate_model(model, seq_len, vocab_size=10, n_batches=10, batch_size=32, device='cuda'):
    """Evaluate model on associative recall classification task"""
    model.eval()
    total_acc = 0
    
    with torch.no_grad():
        for _ in range(n_batches):
            input_seq, targets = generate_associative_recall_batch(seq_len, batch_size, vocab_size, device)
            
            attention_mask = FullMask(seq_len*2 + 1)
            
            output = model(input_seq, attn_mask=attention_mask)
            predictions = output[:, -1].argmax(dim=-1)  # [batch]
            
            acc = (predictions == targets).float().mean()
            total_acc += acc.item()
    
    return total_acc / n_batches

In [7]:
# Run experiments
seq_lengths = [20]
vocab_size = 10
results = {'linear': [], 'softmax': []}

print("\nTraining and evaluating models...")
print("\nSequence Length | Linear Acc | Softmax Acc")
print("-" * 45)

for seq_len in seq_lengths:
    # Reset models
    builder.attention_type = "full"
    softmax_model = builder.get().to('cuda')
    builder.attention_type = "linear"
    linear_model = builder.get().to('cuda')
    
    # Train models
    print(f"\nTraining on sequence length {seq_len}:")
    print("Linear attention model:")
    linear_train_loss = train_model(linear_model, seq_len, vocab_size)
    print("\nSoftmax attention model:")
    softmax_train_loss = train_model(softmax_model, seq_len, vocab_size)
    
    # Evaluate models
    linear_eval_acc = evaluate_model(linear_model, seq_len, vocab_size)
    softmax_eval_acc = evaluate_model(softmax_model, seq_len, vocab_size)
    
    results['linear'].append(linear_eval_acc)
    results['softmax'].append(softmax_eval_acc)
    
    print(f"\n{seq_len:14d} | {linear_eval_acc:.6f} | {softmax_eval_acc:.6f}")

# Create DataFrame with results
results_df = pd.DataFrame({
    'seq_length': seq_lengths,
    'linear_acc': results['linear'],
    'softmax_acc': results['softmax']
})

# Plot results
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(seq_lengths, results['linear'], 'b-o', label='Linear Attention')
plt.plot(seq_lengths, results['softmax'], 'r-o', label='Softmax Attention')
plt.xlabel('Sequence Length')
plt.ylabel('Accuracy')
plt.title('Associative Recall Performance: Linear vs Softmax Attention')
plt.legend()
plt.grid(True)
plt.show()



Training and evaluating models...

Sequence Length | Linear Acc | Softmax Acc
---------------------------------------------

Training on sequence length 20:
Linear attention model:
Epoch 20/1000, Loss: 3.459129, Acc: 0.075195
Epoch 40/1000, Loss: 2.642152, Acc: 0.213867
Epoch 60/1000, Loss: 2.458141, Acc: 0.204102
Epoch 80/1000, Loss: 2.329844, Acc: 0.180664
Epoch 100/1000, Loss: 2.209234, Acc: 0.199219
Epoch 120/1000, Loss: 2.133780, Acc: 0.204102
Epoch 140/1000, Loss: 2.121519, Acc: 0.189453
Epoch 160/1000, Loss: 2.096309, Acc: 0.220703
Epoch 180/1000, Loss: 2.211592, Acc: 0.190430
Epoch 200/1000, Loss: 2.075587, Acc: 0.204102
Epoch 220/1000, Loss: 2.215915, Acc: 0.206055
Epoch 240/1000, Loss: 2.079107, Acc: 0.205078
Epoch 260/1000, Loss: 2.111540, Acc: 0.198242
Epoch 280/1000, Loss: 2.079043, Acc: 0.202148
Epoch 300/1000, Loss: 2.074924, Acc: 0.190430
Epoch 320/1000, Loss: 2.078058, Acc: 0.201172
Epoch 340/1000, Loss: 2.040591, Acc: 0.212891
Epoch 360/1000, Loss: 2.028210, Acc: 0.1

KeyboardInterrupt: 